In [4]:
import numpy as np
import pandas as pd
import os
import json
import csv
import random
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import regularizers
from sklearn.model_selection import train_test_split
from sklearn import preprocessing

# ========================
# Step 1: Load Dataset
# ========================
data = pd.read_csv("/content/news.csv")  # Make sure the file is in the same directory or upload
data = data.drop(["Unnamed: 0"], axis=1)

# ========================
# Step 2: Encode Labels
# ========================
le = preprocessing.LabelEncoder()
data['label'] = le.fit_transform(data['label'])

# ========================
# Step 3: Prepare Inputs
# ========================
embedding_dim = 50
max_length = 54
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"
training_size = 3000
test_portion = 0.1

title = data['title'][:training_size].tolist()
text = data['text'][:training_size].tolist()
labels = data['label'][:training_size].tolist()

# ========================
# Step 4: Tokenize
# ========================
tokenizer = Tokenizer(oov_token=oov_tok)
tokenizer.fit_on_texts(title)
word_index = tokenizer.word_index
vocab_size = len(word_index)

sequences = tokenizer.texts_to_sequences(title)
padded = pad_sequences(sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

split = int(test_portion * training_size)
train_sequences = padded[split:]
train_labels = np.array(labels[split:])
test_sequences = padded[:split]
test_labels = np.array(labels[:split])

# ========================
# Step 5: Load GloVe Embeddings
# ========================
import zipfile
import urllib.request

if not os.path.exists("glove.6B.50d.txt"):
    glove_url = "http://nlp.stanford.edu/data/glove.6B.zip"
    urllib.request.urlretrieve(glove_url, "glove.6B.zip")
    with zipfile.ZipFile("glove.6B.zip", 'r') as zip_ref:
        zip_ref.extractall()

embeddings_index = {}
with open('glove.6B.50d.txt', encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

# ========================
# Step 6: Create Embedding Matrix
# ========================
embedding_matrix = np.zeros((vocab_size + 1, embedding_dim))
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

# ========================
# Step 7: Build Model
# ========================
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size + 1, embedding_dim,
                              weights=[embedding_matrix],
                              input_length=max_length,
                              trainable=False),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# ========================
# Step 8: Train Model
# ========================
model.summary()
history = model.fit(train_sequences, train_labels, epochs=10,
                    validation_data=(test_sequences, test_labels), verbose=2)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │       377,650 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 377,650 (1.44 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 377,650 (1.44 MB)

Epoch 1/10
85/85 - 2s - 26ms/step - accuracy: 0.5356 - loss: 0.6935 - val_accuracy: 0.6633 - val_loss: 0.6836
Epoch 2/10
85/85 - 1s - 11ms/step - accuracy: 0.6348 - loss: 0.6744 - val_accuracy: 0.6833 - val_loss: 0.6606
Epoch 3/10
85/85 - 0s - 3ms/step - accuracy: 0.6741 - loss: 0.6521 - val_accuracy: 0.6567 - val_loss: 0.6351
Epoch 4/10
85/85 - 0s - 4ms/step - accuracy: 0.6822 - loss: 0.6306 - val_accuracy: 0.6833 - val_loss: 0.6181
Epoch 5/10
85/85 - 1s - 7ms/step - accuracy: 0.6896 - loss: 0.6135 - val_accuracy: 0.6900 - val_loss: 0.6005
Epoch 6/10
85/85 - 1s - 7ms/step - accuracy: 0.6937 - loss: 0.6018 - val_accuracy: 0.7033 - val_loss: 0.5886
Epoch 7/10
85/85 - 0s - 4ms/step - accuracy: 0.6963 - loss: 0.5928 - val_accuracy: 0.6900 - val_loss: 0.5783
Epoch 8/10
85/85 - 0s - 3ms/step - accuracy: 0.7004 - loss: 0.5859 - val_accuracy: 0.7067 - val_loss: 0.5770
Epoch 9/10
85/85 - 0s - 4ms/step - accuracy: 0.6989 - loss: 0.5816 - val_accuracy: 0.6733 - val_loss: 0.5641
Epoch 10/10
85/85

In [5]:
# Save the Keras model and tokenizer
import tensorflow as tf
import pickle

# Assuming 'model' is already trained
model.save("fakesniffer_model.h5")

# Save the tokenizer as well
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Model and tokenizer saved successfully.")

Model and tokenizer saved successfully.
